# NLP Assignment 06: Introduction to Natural Language Processing & Text Preprocessing

**Target Text for Analysis:**
> *"Natural Language Processing is an interesting field of Artificial Intelligence! It helps computers understand human language. NLP is used in chatbots, translation, and text analysis."*

### Environment Setup & Imports

In [1]:
# Uncomment below if packages aren't installed yet:
# !pip install nltk spacy
# !python -m spacy download en_core_web_sm

import re
import string
import pandas as pd
import nltk
import spacy

# Download required NLTK datasets
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)

# Load English language model in spaCy
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

text = ("Natural Language Processing is an interesting field of Artificial Intelligence! "
        "It helps computers understand human language. "
        "NLP is used in chatbots, translation, and text analysis.")
print(text)

Natural Language Processing is an interesting field of Artificial Intelligence! It helps computers understand human language. NLP is used in chatbots, translation, and text analysis.


### Question 1: Analyzing Syntax, Semantics, Pragmatics, and Discourse

Breaking down the given passage across the four core levels of linguistic analysis:

- **Syntax (Sentence Structure & Grammar):**
  - *Sentence 1* follows a standard subject–linking verb–complement pattern: **Natural Language Processing** (Subject) + **is** (Auxiliary/Linking Verb) + **an interesting field of Artificial Intelligence** (Noun Phrase).
  - *Sentence 2* uses an active transitive structure: **It** (Subject) + **helps** (Verb) + **computers** (Object) + **understand human language** (Clausal Complement).
  - *Sentence 3* shifts to a **passive voice** construction (*"NLP is used in..."*) followed by a coordinated list of three noun phrases (*chatbots, translation, and text analysis*).

- **Semantics (Literal Meaning):**
  - Literally, the passage defines NLP as a sub-discipline of AI, explains its core goal (enabling machines to interpret human language), and lists practical use cases.
  - Lexical disambiguation happens naturally through context: for instance, the word *"field"* can mean agricultural land or an academic discipline, and surrounding words like *Artificial Intelligence* lock in the academic meaning. Similarly, *"Natural Language Processing"* and *"NLP"* map to the exact same entity.

- **Pragmatics (Contextual Intent & Implied Meaning):**
  - The author's goal is to give a beginner-friendly, upbeat overview—evident from the exclamation mark in the opening sentence.
  - The verb *"understand"* is used figuratively; pragmatically, the reader infers that machines process statistical patterns rather than experiencing human-like comprehension. The passage also assumes baseline familiarity with terms like *AI* and *chatbots*.

- **Discourse (Cohesion Across Sentences):**
  - The paragraph follows a logical flow: **Definition** (Sentence 1) $\rightarrow$ **Function/Purpose** (Sentence 2) $\rightarrow$ **Real-World Applications** (Sentence 3).
  - Cohesive ties bind the text together: the pronoun *"It"* in Sentence 2 refers back to *Natural Language Processing* (anaphora), and the acronym *"NLP"* in Sentence 3 maintains that reference (coreference).

**Quick Code Verification (Syntax, WordNet Senses & Discourse Markers):**

In [2]:
doc = nlp(text)

# Check sentence boundaries (Discourse segmentation)
for i, sent in enumerate(doc.sents, 1):
    print(f"Sentence {i}: {sent.text}")

Sentence 1: Natural Language Processing is an interesting field of Artificial Intelligence!
Sentence 2: It helps computers understand human language.
Sentence 3: NLP is used in chatbots, translation, and text analysis.


In [3]:
# Inspect POS tags and dependency tree via spaCy
syntax = pd.DataFrame([
    {"Token": t.text, "POS": t.pos_, "Dependency": t.dep_, "Head": t.head.text}
    for t in doc if not t.is_space
])
syntax

,Token,POS,Dependency,Head
0,Natural,PROPN,compound,Language
1,Language,PROPN,compound,Processing
2,Processing,NOUN,nsubj,is
3,is,AUX,ROOT,is
4,an,DET,det,field
5,interesting,ADJ,amod,field
6,field,NOUN,attr,is
7,of,ADP,prep,field
8,Artificial,PROPN,compound,Intelligence
9,Intelligence,PROPN,pobj,of


In [4]:
# Check multiple dictionary senses of 'field' in WordNet (Semantics)
from nltk.corpus import wordnet as wn
for syn in wn.synsets("field", pos=wn.NOUN)[:5]:
    print(f"{syn.name():<12} {syn.definition()}")

field.n.01   a piece of land cleared of trees and usually enclosed
battlefield.n.01 a region where a battle is being (or has been) fought
field.n.03   somewhere (away from a studio or office or library or laboratory) where practical work is done or data is collected
discipline.n.01 a branch of knowledge
field.n.05   the space around a radiating body within which its electromagnetic oscillations can exert force on another similar body not in contact with it


In [5]:
# Identify anaphoric pronoun ('It') and acronym ('NLP') linking the sentences
print("Pronouns in text :", [t.text for t in doc if t.pos_ == "PRON"])
print("Acronyms in text :", re.findall(r"\b[A-Z]{2,}\b", text))

Pronouns in text : ['It']
Acronyms in text : ['NLP']


### Question 2: Two Real-World Applications of NLP

1. **Conversational Agents & Virtual Assistants (e.g., Customer Support Bots, Siri, Alexa)**
   - **How it works:** These systems take raw user queries, run preprocessing (tokenization, stopword filtering, lemmatization), classify the user's goal (*intent detection*), and pull out key details like dates or tracking IDs (*named entity recognition*) before generating a response.
   - **Impact:** Automates routine support 24/7, cutting down wait times and operational overhead.

2. **Neural Machine Translation (e.g., Google Translate, DeepL)**
   - **How it works:** Translation models map text from a source language to a target language while preserving meaning, handling differences in word order (syntax), idioms, and surrounding context using deep neural architectures trained on massive parallel corpora.
   - **Impact:** Removes language barriers across global communication, research, and web content.

*(Other notable examples include review sentiment analysis, email spam detection, semantic search, and automatic text summarization.)*

### Question 3: Identifying Potential Noise in the Text

In NLP, **noise** refers to characters, formatting, or low-value tokens that add dimensionality without helping the downstream task. Even though our sample paragraph is well-written, it still has several sources of noise:

- **Punctuation Marks (`!`, `.`, `,`):** Useful for sentence splitting, but typically stripped out in bag-of-words or keyword models.
- **Stopwords (`is`, `an`, `of`, `it`, `in`, `and`):** High-frequency function words that carry very little topical weight on their own.
- **Mixed Capitalization:** Words like *Natural*, *Language*, *Processing*, *Artificial*, and *Intelligence* are capitalized, meaning a case-sensitive tokenizer would treat `Language` and `language` as two separate vocabulary items.
- **Full Form vs. Acronym (`Natural Language Processing` vs. `NLP`):** Both represent the exact same concept, which can split frequency counts if not normalized.
- **Inflected Word Forms (`helps` vs. `help`, `used` vs. `use`):** Morphological variations of the same root word that increase vocabulary size unless stemmed or lemmatized.

In [6]:
# Script to inspect noise elements present in our sample text
punct_found  = [c for c in text if c in string.punctuation]
upper_tokens = re.findall(r"\b[A-Z][a-zA-Z]*\b", text)
acronyms     = re.findall(r"\b[A-Z]{2,}\b", text)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words("english"))
stops_found = [w for w in word_tokenize(text.lower()) if w in stop_words]

print("Punctuation marks     :", punct_found, f"({len(punct_found)} total)")
print("Capitalised words     :", upper_tokens)
print("Acronyms              :", acronyms)
print("Stopwords found       :", stops_found)

Punctuation marks     : ['!', '.', ',', ',', '.'] (5 total)
Capitalised words     : ['Natural', 'Language', 'Processing', 'Artificial', 'Intelligence', 'It', 'NLP']
Acronyms              : ['NLP']
Stopwords found       : ['is', 'an', 'of', 'it', 'is', 'in', 'and']


### Question 4: Removing Punctuation and Stopwords

To clean the text, we apply a standard four-step pipeline:
1. Normalize casing to **lowercase**.
2. Split the text into individual **tokens**.
3. Filter out **punctuation** tokens.
4. Filter out English **stopwords**.

In [7]:
# Step 1 & 2: Lowercase normalization and word tokenization
tokens = word_tokenize(text.lower())
print("Tokens           :", tokens)
print("Number of tokens :", len(tokens))

Tokens           : ['natural', 'language', 'processing', 'is', 'an', 'interesting', 'field', 'of', 'artificial', 'intelligence', '!', 'it', 'helps', 'computers', 'understand', 'human', 'language', '.', 'nlp', 'is', 'used', 'in', 'chatbots', ',', 'translation', ',', 'and', 'text', 'analysis', '.']
Number of tokens : 30


In [8]:
# Step 3: Filter out punctuation marks
no_punct = [t for t in tokens if t not in string.punctuation]
print("After removing punctuation :", no_punct)
print("Number of tokens           :", len(no_punct))

After removing punctuation : ['natural', 'language', 'processing', 'is', 'an', 'interesting', 'field', 'of', 'artificial', 'intelligence', 'it', 'helps', 'computers', 'understand', 'human', 'language', 'nlp', 'is', 'used', 'in', 'chatbots', 'translation', 'and', 'text', 'analysis']
Number of tokens           : 25


In [9]:
# Step 4: Filter out English stopwords
clean_tokens = [t for t in no_punct if t not in stop_words]
print("After removing stopwords :", clean_tokens)
print("Number of tokens         :", len(clean_tokens))

print()
print("Cleaned text:", " ".join(clean_tokens))

After removing stopwords : ['natural', 'language', 'processing', 'interesting', 'field', 'artificial', 'intelligence', 'helps', 'computers', 'understand', 'human', 'language', 'nlp', 'used', 'chatbots', 'translation', 'text', 'analysis']
Number of tokens         : 18

Cleaned text: natural language processing interesting field artificial intelligence helps computers understand human language nlp used chatbots translation text analysis


In [10]:
# Verify exactly which tokens were dropped at each stage
removed_punct = [t for t in tokens if t in string.punctuation]
removed_stops = [t for t in no_punct if t in stop_words]
print("Punctuation removed :", removed_punct)
print("Stopwords removed   :", removed_stops)

Punctuation removed : ['!', '.', ',', ',', '.']
Stopwords removed   : ['is', 'an', 'of', 'it', 'is', 'in', 'and']


### Question 5: Text Preprocessing Using NLTK vs. spaCy

- **NLTK (Natural Language Toolkit):** Modular and great for learning/experimentation. You manually chain separate tools together (`word_tokenize`, `stopwords`, `PorterStemmer`, `WordNetLemmatizer`, `pos_tag`), giving fine-grained control over each transformation.
- **spaCy:** Built for speed and production pipelines. Calling `nlp(text)` runs tokenization, tagging, parsing, and lemmatization in a single pass, exposing token properties directly (e.g., `.lemma_`, `.pos_`, `.is_stop`, `.is_punct`).

**1. Preprocessing Pipeline with NLTK:**

In [11]:
from nltk.tokenize import sent_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag

print("Sentences    :", sent_tokenize(text))
print()

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Run POS tagging on full context first, then filter out punctuation and stopwords
tagged = [(w.lower(), tag) for w, tag in pos_tag(word_tokenize(text))
          if w.lower() not in stop_words and w not in string.punctuation]

nltk_table = pd.DataFrame({
    "Token": [w for w, _ in tagged],
    "Stem (Porter)": [stemmer.stem(w) for w, _ in tagged],
    "Lemma (WordNet)": [lemmatizer.lemmatize(w) for w, _ in tagged],
    "POS tag": [tag for _, tag in tagged],
})
nltk_table

Sentences    : ['Natural Language Processing is an interesting field of Artificial Intelligence!', 'It helps computers understand human language.', 'NLP is used in chatbots, translation, and text analysis.']



,Token,Stem (Porter),Lemma (WordNet),POS tag
0,natural,natur,natural,JJ
1,language,languag,language,NNP
2,processing,process,processing,NNP
3,interesting,interest,interesting,JJ
4,field,field,field,NN
5,artificial,artifici,artificial,JJ
6,intelligence,intellig,intelligence,NN
7,helps,help,help,VBZ
8,computers,comput,computer,NNS
9,understand,understand,understand,VBP


**2. Preprocessing Pipeline with spaCy:**

In [12]:
doc = nlp(text)

# Single-pass cleaning: lowercase lemmas without punctuation, stopwords, or whitespace
spacy_tokens = [t.lemma_.lower() for t in doc if not t.is_punct and not t.is_stop and not t.is_space]
print("spaCy preprocessed tokens:", spacy_tokens)

spacy_table = pd.DataFrame([
    {"Token": t.text, "Lemma": t.lemma_, "POS": t.pos_, "Is stopword": t.is_stop, "Is punctuation": t.is_punct}
    for t in doc
])
spacy_table

spaCy preprocessed tokens: ['natural', 'language', 'processing', 'interesting', 'field', 'artificial', 'intelligence', 'help', 'computer', 'understand', 'human', 'language', 'nlp', 'chatbot', 'translation', 'text', 'analysis']


,Token,Lemma,POS,Is stopword,Is punctuation
0,Natural,Natural,PROPN,False,False
1,Language,Language,PROPN,False,False
2,Processing,processing,NOUN,False,False
3,is,be,AUX,True,False
4,an,an,DET,True,False
5,interesting,interesting,ADJ,False,False
6,field,field,NOUN,False,False
7,of,of,ADP,True,False
8,Artificial,Artificial,PROPN,False,False
9,Intelligence,Intelligence,PROPN,False,False


**Key Takeaways: NLTK vs. spaCy**

| Aspect | NLTK | spaCy |
|---|---|---|
| **Architecture** | Modular toolkit (step-by-step functions) | End-to-end object-oriented pipeline (`nlp(text)`) |
| **Tokenization** | Explicit via `word_tokenize()` | Automatic when creating a `Doc` object |
| **Stopwords & Punctuation** | Manual list/set lookup | Built-in boolean flags (`token.is_stop`, `token.is_punct`) |
| **Stemming** | Supported (`PorterStemmer`, `SnowballStemmer`) | Not included (focuses on accurate lemmatization) |
| **Lemmatization** | `WordNetLemmatizer` (requires manual POS tags for verbs/adjectives) | Context-aware `token.lemma_` out of the box |
| **Performance & Use Case** | Slower; best for teaching, prototyping, and custom linguistic rules | Fast (Cython-backed); best for production workflows |

### Question 6: Future Directions of NLP

1. **Smarter, Reasoning-Capable LLMs:** Foundation models are evolving beyond simple text generation toward multi-step reasoning, tool use, and deep integration into everyday software.
2. **Low-Resource & Multilingual Inclusion:** Expanding high-accuracy NLP beyond English to underrepresented and regional languages (such as Urdu and South Asian languages) where labeled datasets are scarce.
3. **Unified Multimodal Models:** Seamless processing across text, images, audio, and video—enabling systems that can reason over documents, speech, and visual scenes together.
4. **Compact & Edge/On-Device Models:** Quantization and distillation techniques allowing smaller language models to run locally on phones and laptops with lower latency, energy cost, and stronger data privacy.
5. **Hallucination Reduction, Safety & Ethics:** Grounding outputs in verified sources (e.g., RAG), mitigating social biases, and preventing synthetic misinformation.
6. **Explainability & Domain Specialization:** Building interpretable, trustworthy models tailored for high-stakes sectors like healthcare, legal analysis, and education.

### Summary

In this notebook, we examined the given passage across four linguistic layers (**syntax, semantics, pragmatics, and discourse**), identified common text noise, and built a cleaning pipeline that reduced the original **30 tokens** down to **18 core content words**. Finally, we demonstrated and contrasted how **NLTK** and **spaCy** handle tokenization, tagging, stemming, and lemmatization.